# Etapa 3 · Punto 4 — Ranking de las recomendaciones

**Objetivo.** Asignar un **puntaje final** a los candidatos que generan los algoritmos de grafos del punto 3 y producir
un **Top-K** ordenado. El puntaje combina la señal estructural del grafo con la **información de preferencias de
usuarios** que se integró en el punto 2, es decir, las reseñas y calificaciones de CritiqueBrainz.

### Metodología: ranking en dos etapas

```
preferencia ──► [Punto 3] Personalized PageRank ──► 200 candidatos ──► [Punto 4] puntaje final ──► Top-K (diversidad)
                                                                         ▲
                       reseñas de CritiqueBrainz (punto 2) ─ valoración, afinidad textual
```

El puntaje final de cada candidato `c` es una combinación lineal de cuatro señales normalizadas en [0, 1] dentro del
conjunto de candidatos:

$$\text{score}(c) = w_G\,G(c) + w_V\,V(c) + w_A\,A(c) + w_E\,E(c), \qquad \textstyle\sum w = 1$$

| Señal | Fuente | Definición | Si no hay datos |
|---|---|---|---|
| **G**: relevancia en el grafo | Punto 3 (PPR) | `log(PPR)`, escalado min-max entre los candidatos | — |
| **V**: valoración de usuarios | Punto 2 (reseñas) | Promedio bayesiano de las valoraciones de las reseñas que alcanzan la canción, ya sean de la canción, su álbum o su artista. Cuenta la calificación explícita (1–5) o, si no existe, una **estimada a partir del texto** | Valor neutro (media global) |
| **A**: afinidad textual | Punto 2 (embeddings) | Coseno entre el **perfil** de la preferencia y el vector de las reseñas del candidato | Media de los candidatos |
| **E**: misma escena | Punto 3 (Louvain) | 1 si el artista del candidato está en la comunidad principal de la preferencia | 0 |

Después del puntaje se aplica la misma regla de diversidad del punto 3: máximo 2 canciones por artista.

**Selección de pesos (sección 5).** Se hace una búsqueda en rejilla sobre el simplex, separando validación y prueba,
con una regla explícita: **maximizar el respaldo de preferencias del Top-10 sin perder relevancia** frente al ranking
que usa solo el grafo. La relevancia se mide ocultando relaciones, igual que en el punto 3.

> Requiere los notebooks 01, 02 y 03 ejecutados y `entrega_3/.env`. La generación de candidatos se importa de
> `src/recomendador_grafo.py`, que empaqueta el código del notebook 03.

## 0. Entorno

In [1]:
%pip -q install "neo4j>=5.26,<7" "networkx>=3.2" "pandas>=2" "scikit-learn>=1.3" "sentence-transformers>=3,<6"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, time, random, itertools
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 220)

DATA_DIR = Path(os.environ.get('DATA_DIR', '../data'))
sys.path.insert(0, str(Path('../src').resolve()))
import recomendador_grafo as rg

N_CANDIDATOS = 200        # candidatos que entrega el punto 3 a esta etapa
K_PRIOR = 2.0             # fuerza del prior bayesiano de la valoración (equivale a 2 reseñas "promedio")
CONF_ESTIMADA = 0.5       # peso de una valoración estimada desde el texto, frente a 1.0 de una calificación explícita
PASO_REJILLA = 0.1        # resolución de la búsqueda de pesos
SEMILLA = 42

In [3]:
# Conexión a Neo4j (credenciales desde entrega_3/.env; reintentos ante cortes de Aura)
from neo4j import GraphDatabase
from neo4j.exceptions import DriverError, ServiceUnavailable, SessionExpired
import logging
logging.getLogger('neo4j').setLevel(logging.CRITICAL)

def cargar_env(ruta):
    for linea in ruta.read_text(encoding='utf-8').splitlines():
        if '=' in linea and not linea.lstrip().startswith('#'):
            k, v = linea.split('=', 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")

ARCHIVO_ENV = next((p for p in [Path('../.env'), Path('.env'), DATA_DIR / '.env'] if p.exists()), None)
if ARCHIVO_ENV is None:
    raise FileNotFoundError('No se encontró entrega_3/.env. Cópialo desde .env.example y completa las credenciales.')
cargar_env(ARCHIVO_ENV)
NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD = (os.environ[k] for k in ('NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD'))
NEO4J_DATABASE = os.environ.get('NEO4J_DATABASE') or None

def conectar(intentos=15, espera=5):
    for i in range(1, intentos + 1):
        d = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD), connection_timeout=15,
                                 max_transaction_retry_time=120, notifications_min_severity='OFF')
        try:
            d.verify_connectivity()
            return d
        except (ServiceUnavailable, SessionExpired) as e:
            d.close()
            if i == intentos:
                raise
            print(f'Aura no disponible ({type(e).__name__}); reintento {i}/{intentos - 1} en {espera}s...')
            time.sleep(espera)

def sesion():
    global driver
    try:
        return driver.session(database=NEO4J_DATABASE)
    except DriverError:
        driver = conectar()
        return driver.session(database=NEO4J_DATABASE)

def con_reintentos(fn, intentos=15, espera=5):
    global driver
    for i in range(1, intentos + 1):
        try:
            with sesion() as session:
                return fn(session)
        except (ServiceUnavailable, SessionExpired) as e:
            if i == intentos:
                raise
            print(f'Conexión perdida ({type(e).__name__}); reconectando {i}/{intentos - 1} en {espera}s...')
            time.sleep(espera)
            try:
                driver.close()
            except Exception:
                pass
            driver = conectar()

def run_cypher(query, **parameters):
    return con_reintentos(lambda s: [dict(r) for r in s.run(query, **parameters)])

def cargar_en_lotes(nombre, query, rows, batch_size=1000):
    t = time.time()
    for i in range(0, len(rows), batch_size):
        lote = rows[i:i + batch_size]
        con_reintentos(lambda s: s.execute_write(lambda tx: tx.run(query, rows=lote).consume()))
    print(f'✓ {nombre:<40} {len(rows):>8,}  ({time.time() - t:.1f}s)')

driver = conectar()
print('Conexión establecida con', driver.get_server_info().agent)
assert run_cypher('MATCH (a:Artist) WHERE a.community IS NOT NULL RETURN count(a) AS n')[0]['n'] > 0, \
    'Falta el punto 3: ejecuta primero 03_recomendacion_algoritmos_grafos.ipynb'

Conexión establecida con Neo4j/5.27-aura


## 1. Candidatos del punto 3

`GrafoRecomendador` proyecta el grafo desde Neo4j, recalcula las comunidades de Louvain (con la misma semilla) y expone
`candidatos(semillas, n)`. Esa función devuelve las `n` canciones con mayor Personalized PageRank, sin las ya conocidas
y sin duplicados del MSD.

In [4]:
t0 = time.time()
gr = rg.GrafoRecomendador(run_cypher)
print(f'Grafo: {gr.ctx["G"].number_of_nodes():,} nodos, {gr.ctx["G"].number_of_edges():,} aristas | '
      f'{len(gr.ctx["comms"])} comunidades Louvain  ({time.time() - t0:.0f}s)')

Grafo: 130,031 nodos, 136,721 aristas | 133 comunidades Louvain  (62s)


## 2. Señales de preferencia de usuarios (punto 2)

Para cada reseña se recuperan de Neo4j su autor, la entidad reseñada y la calificación, y se calcula su **vector**: el
promedio normalizado de los embeddings de sus chunks, que ya están guardados en el índice vectorial del punto 2.

In [5]:
RESENAS = {f['review']: f for f in run_cypher('''
    MATCH (u:User)<-[:WRITTEN_BY]-(r:Review)-[:REVIEWS_RECORDING|REVIEWS_ALBUM|REVIEWS_ARTIST]->(e)
    RETURN r.uri AS review, u.uri AS usuario, e.uri AS entidad, labels(e)[0] AS tipo, r.rating AS rating''')}
acum = defaultdict(list)
for f in run_cypher('MATCH (c:Chunk)-[:PART_OF]->(r:Review) RETURN r.uri AS review, c.embedding AS e'):
    acum[f['review']].append(np.asarray(f['e'], dtype=np.float32))
EMB = {}
for r, vs in acum.items():
    m = np.mean(vs, axis=0)
    EMB[r] = m / np.linalg.norm(m)

canciones_album, canciones_artista = defaultdict(set), defaultdict(set)
for f in gr.datos['APPEARS_ON']:
    canciones_album[f['o']].add(f['s'])
for f in gr.datos['SUNG_BY']:
    canciones_artista[f['o']].add(f['s'])
def canciones_de(entidad, tipo):
    return {entidad} if tipo == 'Track' else canciones_album[entidad] if tipo == 'Album' else canciones_artista[entidad]

RESENAS_CANCION = defaultdict(set)        # canción -> reseñas que la alcanzan (de la canción, su álbum o su artista)
RESENAS_USUARIO = defaultdict(set)
for r, f in RESENAS.items():
    for t in canciones_de(f['entidad'], f['tipo']):
        RESENAS_CANCION[t].add(r)
    RESENAS_USUARIO[f['usuario']].add(r)
print(f'Reseñas: {len(RESENAS):,} | con texto (vector): {len(EMB):,} | con calificación: '
      f'{sum(f["rating"] is not None for f in RESENAS.values())} | canciones alcanzadas: {len(RESENAS_CANCION):,}')

Reseñas: 2,006 | con texto (vector): 1,855 | con calificación: 303 | canciones alcanzadas: 3,206


### 2.1 Valoración `V`: calificaciones explícitas y estimadas desde el texto

Solo ~15 % de las reseñas trae calificación de 1 a 5 estrellas. La mayoría son críticas de la BBC con texto pero sin
estrellas. Para no desperdiciar esa opinión, se entrena una **regresión ridge** que predice las estrellas a partir del
vector de la reseña, usando las reseñas que tienen texto y calificación. Se valida con validación cruzada de 5
particiones antes de usarla.

In [6]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_predict
from scipy.stats import spearmanr

etiquetadas = sorted(r for r, f in RESENAS.items() if f['rating'] is not None and r in EMB)
X = np.array([EMB[r] for r in etiquetadas])
y = np.array([RESENAS[r]['rating'] for r in etiquetadas], dtype=float)
modelo_valoracion = RidgeCV(alphas=np.logspace(-2, 3, 12))
pred_cv = cross_val_predict(modelo_valoracion, X, y, cv=KFold(5, shuffle=True, random_state=SEMILLA))
display(pd.DataFrame({'reseñas': [len(y)],
                      'Spearman (CV)': [round(spearmanr(pred_cv, y).correlation, 3)],
                      'MAE ridge (CV)': [round(np.abs(pred_cv - y).mean(), 3)],
                      'MAE predecir la media': [round(np.abs(y - y.mean()).mean(), 3)]}, index=['estimación de estrellas']))
modelo_valoracion.fit(X, y)

VALORACION = {}                            # reseña -> (estrellas, confianza)
for r, f in RESENAS.items():
    if f['rating'] is not None:
        VALORACION[r] = (float(f['rating']), 1.0)
    elif r in EMB:
        VALORACION[r] = (float(np.clip(modelo_valoracion.predict(EMB[r][None])[0], 1, 5)), CONF_ESTIMADA)
MU = sum(v * w for v, w in VALORACION.values()) / sum(w for v, w in VALORACION.values())
print(f'Valoraciones: {sum(w == 1 for _, w in VALORACION.values())} explícitas + '
      f'{sum(w < 1 for _, w in VALORACION.values())} estimadas | media global (prior) μ = {MU:.2f}')

def valoracion(t, excluir=frozenset()):
    # Promedio bayesiano: (K·μ + Σ w·v) / (K + Σ w). Sin reseñas devuelve μ, un valor neutro
    num, den, n = K_PRIOR * MU, K_PRIOR, 0
    for r in RESENAS_CANCION.get(t, ()):
        if r in VALORACION and r not in excluir:
            v, w = VALORACION[r]
            num, den, n = num + w * v, den + w, n + 1
    return num / den, n

,reseñas,Spearman (CV),MAE ridge (CV),MAE predecir la media
estimación de estrellas,153,0.418,0.711,0.77


Valoraciones: 303 explícitas + 1702 estimadas | media global (prior) μ = 4.07


**Parámetros de `V`:**
- **`K_PRIOR = 2`:** una canción con una sola reseña de 5 estrellas no supera automáticamente a una con diez reseñas de
  4,5. El promedio se *encoge* hacia la media global μ en proporción a la evidencia.
- **`CONF_ESTIMADA = 0,5`:** una estrella estimada vale la mitad que una explícita, porque tiene error. La ridge reduce
  el error frente a predecir la media, pero no lo elimina (ver tabla).
- **μ:** se calcula con *todas* las valoraciones, explícitas y estimadas. Si se usaran solo las explícitas (media ≈ 4,14,
  frente a ≈ 4,05 de las estimadas), tener reseñas penalizaría a una canción frente a no tenerlas.

### 2.2 Afinidad textual `A`: perfil de la preferencia frente a las reseñas del candidato

- **Perfil:** es el promedio de los vectores de reseña asociados a las semillas. Para un usuario son sus propias
  reseñas; para una canción, artista o álbum, las reseñas que los alcanzan. Para un texto libre, el perfil es el
  embedding de la consulta.
- **Vector del candidato:** es el promedio de las reseñas que lo alcanzan, sin contar las del propio usuario.

**¿Es una señal de relevancia?** Se valida con un experimento independiente. Entre canciones con reseña, se compara la
similitud textual de los pares `SIMILAR_TO` reales con la de pares al azar. Se excluyen los pares que comparten reseña,
porque serían triviales.

In [7]:
from sklearn.metrics import roc_auc_score

def vector_cancion(t, excluir=frozenset()):
    vs = [EMB[r] for r in RESENAS_CANCION.get(t, ()) if r in EMB and r not in excluir]
    if not vs:
        return None
    m = np.mean(vs, axis=0)
    return m / np.linalg.norm(m)

VEC = {t: v for t in RESENAS_CANCION if (v := vector_cancion(t)) is not None}
pares = [(f['s'], f['o']) for f in gr.datos['SIMILAR_TO']
         if f['s'] in VEC and f['o'] in VEC and not (RESENAS_CANCION[f['s']] & RESENAS_CANCION[f['o']])]
rng, con_vec, negativos = random.Random(SEMILLA), sorted(VEC), []
while len(negativos) < 5 * len(pares):
    a, b = rng.choice(con_vec), rng.choice(con_vec)
    if a != b and not (RESENAS_CANCION[a] & RESENAS_CANCION[b]):
        negativos.append((a, b))
sims = [float(VEC[a] @ VEC[b]) for a, b in pares + negativos]
print(f'Afinidad textual — pares SIMILAR_TO con texto en ambos: {len(pares)} | '
      f'AUC frente a pares al azar = {roc_auc_score([1] * len(pares) + [0] * len(negativos), sims):.3f} (0,5 = azar)')

Afinidad textual — pares SIMILAR_TO con texto en ambos: 67 | AUC frente a pares al azar = 0.674 (0,5 = azar)


In [8]:
_modelo_texto = None
def perfil(semillas, excluir=frozenset(), texto=None):
    global _modelo_texto
    if texto:
        if _modelo_texto is None:
            from sentence_transformers import SentenceTransformer
            _modelo_texto = SentenceTransformer('intfloat/multilingual-e5-small')
        return _modelo_texto.encode(f'query: {texto}', normalize_embeddings=True)
    vs = []
    for n in semillas:
        t = gr.tipo.get(n)
        if t == 'User':
            rs = RESENAS_USUARIO[n]
        elif t == 'Track':
            rs = RESENAS_CANCION.get(n, set())
        elif t in ('Album', 'Artist'):
            rs = {r for r, f in RESENAS.items() if f['entidad'] == n}
            if t == 'Artist':
                rs |= {r for c in canciones_artista[n] for r in RESENAS_CANCION.get(c, ())}
        else:
            rs = set()
        vs += [EMB[r] for r in rs if r in EMB and r not in excluir]
    if not vs:
        return None
    m = np.mean(vs, axis=0)
    return m / np.linalg.norm(m)

**Lectura.** La similitud entre las reseñas de dos canciones distingue claramente las canciones similares de las
tomadas al azar (AUC muy por encima de 0,5). Por lo tanto `A` no solo refleja el gusto: también aporta **relevancia**
que el grafo no ve.

## 3. Puntaje final y Top-K

Las cuatro señales se escalan con min-max **dentro del conjunto de candidatos**. Así los pesos expresan importancia
relativa en la misma escala. Si a un candidato le falta `A` (no tiene reseñas con texto), toma la media de los
candidatos, de modo que no se le favorece ni se le penaliza. Si la preferencia no tiene perfil textual, `A` es
constante y no altera el orden.

In [9]:
SENALES = ['G', 'V', 'A', 'E']

def minmax(x):
    x = np.asarray(x, dtype=float)
    lo, hi = np.min(x), np.max(x)
    return np.zeros_like(x) if hi - lo < 1e-12 else (x - lo) / (hi - lo)

def matriz_senales(semillas, cands, ctx=None, excluir=frozenset(), usuario=None, texto=None):
    ctx = ctx or gr.ctx
    prof = perfil(semillas, excluir, texto)
    esc = gr.escena(semillas, ctx)
    principal = esc.most_common(1)[0][0] if esc else None
    excl_vec = set(excluir) | (RESENAS_USUARIO[usuario] if usuario else set())
    estrellas, evidencia, afin, escena_ = [], [], [], []
    for t, _ in cands:
        v, n = valoracion(t, excluir)
        estrellas.append(v); evidencia.append(n)
        vt = vector_cancion(t, excl_vec) if prof is not None else None
        afin.append(np.nan if vt is None else float(prof @ vt))
        escena_.append(1.0 if principal is not None and ctx['com'].get(gr.artista_de[t]) == principal else 0.0)
    afin = np.array(afin)
    afin = np.zeros(len(afin)) if np.isnan(afin).all() else np.where(np.isnan(afin), np.nanmean(afin), afin)
    F = np.c_[minmax(np.log([s for _, s in cands])), minmax(estrellas), minmax(afin), escena_]
    extra = pd.DataFrame({'estrellas (V)': estrellas, 'reseñas': evidencia, 'afinidad (coseno)': afin})
    return F, extra

def top_indices(cands, score, k=10, max_por_artista=rg.MAX_POR_ARTISTA):
    orden, salida, cuenta = np.argsort(-score, kind='stable'), [], Counter()
    for i in orden:
        a = gr.artista_de[cands[i][0]]
        if cuenta[a] < max_por_artista:
            salida.append(i); cuenta[a] += 1
        if len(salida) == k:
            break
    return salida

## 4. Selección de los pesos

**Datos de validación.** Son los mismos experimentos de ocultamiento del punto 3, sin fuga de información: las
comunidades, PPR y las señales se recalculan sin lo oculto.

- **Canción → similares (relevancia):** 200 canciones a las que se les oculta el 30 % de sus `SIMILAR_TO`. Se mide
  recall@10 y recall@50 de las ocultas.
- **Usuario → reseña oculta:** 139 usuarios a los que se les oculta una entidad reseñada. También se oculta esa reseña
  en `V` y `A`.

Los casos se dividen al azar en **validación (50 %)** y **prueba (50 %)**.

**Métricas:**
- **Relevancia `R`:** recall@10 y recall@50 relativos al ranking que usa solo el grafo (`w = (1, 0, 0, 0)`), promediados.
- **Respaldo de preferencias `Q`:** fracción del Top-10 formada por canciones con reseñas cuya valoración es ≥ μ, es
  decir, recomendaciones respaldadas por opiniones favorables de usuarios.

**Regla de selección (explícita):** entre todas las combinaciones de la rejilla (paso 0,1, `w_G ≥ 0,1`), se elige la de
**mayor `Q` con `R ≥ 1`** en validación. Es decir: se incorpora toda la información de preferencias que se pueda **sin
perder relevancia** frente al grafo solo. Los empates se resuelven por mayor `R`. Después se reporta el resultado en
**prueba**.

In [10]:
t0 = time.time()
rng = random.Random(SEMILLA)
# --- Tarea 1: canción -> similares ocultas
SIMILARES = defaultdict(list)
for f in gr.datos['SIMILAR_TO']:
    SIMILARES[f['s']].append(f['o'])
semillas_sim = sorted(t for t, v in SIMILARES.items() if len(v) >= 6)
rng.shuffle(semillas_sim)
semillas_sim = semillas_sim[:200]
ocultas = {}
for t in semillas_sim:
    v = sorted(SIMILARES[t]); rng.shuffle(v)
    ocultas[t] = set(v[:max(2, len(v) * 3 // 10)])
ctx_sim = gr.contexto({p for t, hs in ocultas.items() for h in hs for p in ((t, h), (h, t))})
CASOS = []
for t in semillas_sim:
    conocidas = {t} | (set(SIMILARES[t]) - ocultas[t])
    cands = gr.candidatos({t: 1.0}, N_CANDIDATOS, ctx_sim, conocidas)
    F, extra = matriz_senales({t: 1.0}, cands, ctx_sim)
    CASOS.append({'tarea': 'similares', 'cands': cands, 'F': F, 'extra': extra, 'objetivo': ocultas[t]})

# --- Tarea 2: usuario -> entidad reseñada oculta
por_usuario = defaultdict(list)
for f in RESENAS.values():
    por_usuario[f['usuario']].append(f)
def artista_de_entidad(f):
    e = f['entidad']
    return gr.artista_de.get(e) if f['tipo'] == 'Track' else e if f['tipo'] == 'Artist' else next(iter(gr.artistas_de(e)), None)
oculta_usuario = {}
for u, fs in sorted(por_usuario.items()):
    if len({artista_de_entidad(f) for f in fs}) >= 3:
        f = rng.choice(sorted(fs, key=lambda x: x['review']))
        if len({artista_de_entidad(g) for g in fs if g is not f} - {artista_de_entidad(f)}) >= 2:
            oculta_usuario[u] = f
ctx_usr = gr.contexto({(u, f['entidad']) for u, f in oculta_usuario.items()})
for u, f in oculta_usuario.items():
    conocidas = set().union(*(canciones_de(g['entidad'], g['tipo']) for g in por_usuario[u] if g is not f))
    cands = gr.candidatos({u: 1.0}, N_CANDIDATOS, ctx_usr, conocidas)
    F, extra = matriz_senales({u: 1.0}, cands, ctx_usr, excluir={f['review']}, usuario=u)
    CASOS.append({'tarea': 'usuario', 'cands': cands, 'F': F, 'extra': extra,
                  'objetivo': canciones_de(f['entidad'], f['tipo'])})
print(f'{len(CASOS)} casos ({len(semillas_sim)} canciones + {len(oculta_usuario)} usuarios) en {time.time() - t0:.0f}s')

339 casos (200 canciones + 139 usuarios) en 75s


In [11]:
def evaluar(w, casos):
    m = defaultdict(list)
    for c in casos:
        score = c['F'] @ np.asarray(w)
        top10, top50 = top_indices(c['cands'], score, 10), top_indices(c['cands'], score, 50)
        ids = lambda idx: {c['cands'][i][0] for i in idx}
        if c['tarea'] == 'similares':
            m['recall@10'].append(len(c['objetivo'] & ids(top10)) / len(c['objetivo']))
            m['recall@50'].append(len(c['objetivo'] & ids(top50)) / len(c['objetivo']))
        else:
            m['usuario hit@50'].append(bool(c['objetivo'] & ids(top50)))
        ex = c['extra'].iloc[top10]
        m['Q (respaldo)'].append(float(((ex['reseñas'] > 0) & (ex['estrellas (V)'] >= MU)).mean()))
        m['con reseñas'].append(float((ex['reseñas'] > 0).mean()))
        m['estrellas medias'].append(float(ex['estrellas (V)'].mean()))
    return {k: float(np.mean(v)) for k, v in m.items()}

orden = list(range(len(CASOS)))
random.Random(7).shuffle(orden)
VALIDACION = [CASOS[i] for i in orden[:len(orden) // 2]]
PRUEBA = [CASOS[i] for i in orden[len(orden) // 2:]]

pasos = np.round(np.arange(0, 1 + 1e-9, PASO_REJILLA), 2)
REJILLA = [w for w in itertools.product(pasos, repeat=4) if abs(sum(w) - 1) < 1e-9 and w[0] >= 0.1]
t0 = time.time()
res = pd.DataFrame([{'w': tuple(float(x) for x in w), **evaluar(w, VALIDACION)} for w in REJILLA])
base = res[res.w == (1.0, 0.0, 0.0, 0.0)].iloc[0]
res['R'] = (res['recall@10'] / base['recall@10'] + res['recall@50'] / base['recall@50']) / 2
print(f'{len(REJILLA)} combinaciones evaluadas en validación ({time.time() - t0:.0f}s)')

def mejor(umbral):
    return res[res.R >= umbral].sort_values(['Q (respaldo)', 'R'], ascending=False).iloc[0]
seleccion = mejor(1.0)
W = {s: float(x) for s, x in zip(SENALES, seleccion.w)}
print('Pesos seleccionados:', W)

frontera = pd.DataFrame([{'R mínima': u, 'pesos (G, V, A, E)': mejor(u).w, 'R': mejor(u).R,
                          'Q (respaldo)': mejor(u)['Q (respaldo)']} for u in (1.0, 0.975, 0.95, 0.9, 0.0)])
display(frontera.set_index('R mínima').round(3))

220 combinaciones evaluadas en validación (6s)
Pesos seleccionados: {'G': 0.2, 'V': 0.1, 'A': 0.7, 'E': 0.0}


,"pesos (G, V, A, E)",R,Q (respaldo)
R mínima,,,
1.000,"(0.2, 0.1, 0.7, 0.0)",1.043,0.232
0.975,"(0.2, 0.3, 0.5, 0.0)",0.976,0.354
0.950,"(0.1, 0.7, 0.2, 0.0)",0.955,0.569
0.900,"(0.1, 0.9, 0.0, 0.0)",0.940,0.586
0.000,"(0.1, 0.9, 0.0, 0.0)",0.940,0.586


La **frontera** muestra el compromiso entre ambas métricas. Si se acepta perder algo de relevancia, se puede subir más
el respaldo de preferencias, pero entonces el ranking deja de ser fiel al grafo. La regla elegida (`R ≥ 1`) no acepta
esa pérdida.

### 4.1 Resultado en prueba y ablación

In [12]:
w_sel = tuple(W[s] for s in SENALES)
def sin(senal):
    w = dict(W)
    w['G'] += w[senal]; w[senal] = 0.0
    return tuple(w[s] for s in SENALES)
variantes = {'solo grafo (punto 3)': (1.0, 0.0, 0.0, 0.0), 'pesos iguales': (0.25, 0.25, 0.25, 0.25),
             'SELECCIONADO': w_sel, 'sin V (valoración)': sin('V'), 'sin A (afinidad)': sin('A'), 'sin E (escena)': sin('E')}
tabla = pd.DataFrame({n: {'pesos (G, V, A, E)': tuple(round(x, 2) for x in w), **evaluar(w, PRUEBA)}
                      for n, w in variantes.items()}).T
display(tabla[['pesos (G, V, A, E)', 'recall@10', 'recall@50', 'usuario hit@50', 'Q (respaldo)', 'con reseñas',
               'estrellas medias']].style.format(precision=3))

,"pesos (G, V, A, E)",recall@10,recall@50,usuario hit@50,Q (respaldo),con reseñas,estrellas medias
solo grafo (punto 3),"(1.0, 0.0, 0.0, 0.0)",0.043,0.215,0.000,0.091,0.171,4.077
pesos iguales,"(0.25, 0.25, 0.25, 0.25)",0.055,0.218,0.000,0.158,0.215,4.093
SELECCIONADO,"(0.2, 0.1, 0.7, 0.0)",0.045,0.213,0.014,0.210,0.353,4.091
sin V (valoración),"(0.3, 0.0, 0.7, 0.0)",0.041,0.213,0.000,0.141,0.285,4.076
sin A (afinidad),"(0.9, 0.1, 0.0, 0.0)",0.043,0.215,0.000,0.114,0.175,4.084
sin E (escena),"(0.2, 0.1, 0.7, 0.0)",0.045,0.213,0.014,0.210,0.353,4.091


Los pesos y parámetros elegidos se guardan en `data/ranking_config.json`. El sistema de recomendación del punto 5
(`src/ranking.py`) los lee de ahí, así que usa exactamente la configuración seleccionada aquí.

In [13]:
import json
CONFIG = {'pesos': W, 'n_candidatos': N_CANDIDATOS, 'k_prior': K_PRIOR, 'conf_estimada': CONF_ESTIMADA,
          'seleccion': {'regla': 'máximo Q (respaldo) con R >= 1 en validación', 'paso_rejilla': PASO_REJILLA,
                        'R_validacion': round(float(seleccion.R), 4), 'Q_validacion': round(float(seleccion['Q (respaldo)']), 4)}}
(DATA_DIR / 'ranking_config.json').write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

{
  "pesos": {
    "G": 0.2,
    "V": 0.1,
    "A": 0.7,
    "E": 0.0
  },
  "n_candidatos": 200,
  "k_prior": 2.0,
  "conf_estimada": 0.5,
  "seleccion": {
    "regla": "máximo Q (respaldo) con R >= 1 en validación",
    "paso_rejilla": 0.1,
    "R_validacion": 1.0426,
    "Q_validacion": 0.232
  }
}


**Lectura de la selección:**

- **Relevancia y respaldo:** en prueba, el ranking final mantiene la relevancia del grafo y **aumenta mucho la fracción
  del Top-10 respaldada por opiniones favorables** de usuarios, además de la fracción con reseñas. Esas reseñas son
  justamente la evidencia que necesita el componente RAG para explicar las recomendaciones.
- **El peso de `G` no mide toda su influencia.** El grafo ya decide *qué* 200 canciones llegan a esta etapa. Además, la
  mayoría de candidatos no tiene reseñas, así que su `V` y su `A` son neutros (constantes) y su orden entre sí lo decide
  solo `G`. Lo que hacen `V` y `A` es reubicar a los candidatos *con* evidencia de usuarios.
- **Afinidad `A`:** tiene el mayor peso porque es la única señal de preferencia que también aporta relevancia (AUC de la
  sección 2.2).
- **Valoración `V`:** su peso es menor porque premia la calidad percibida, pero no la cercanía al gusto.
- **Escena `E`:** recibe 0. En el punto 3 Louvain ya mostró su valor como generador y para explicar, pero como término
  del puntaje no añade relevancia: PPR ya incorpora esa estructura.
- **Tarea de usuarios:** casi no tiene aciertos en ningún método, como ya se vio en el punto 3. Casi todos los usuarios
  son críticos que reseñan discos de todos los géneros y cada disco tiene una sola reseña. Por eso no se usa para
  elegir la relevancia y se reporta solo como referencia.

## 5. Top-K para una preferencia

`top_k(...)` recibe la misma preferencia que en el punto 3 (`canciones`, `artistas`, `tags`, `usuario`, `texto`) y
devuelve la lista ordenada con el puntaje final, sus componentes y la evidencia de usuarios. También muestra cuánto
subió o bajó cada canción respecto al orden que usa solo el grafo.

In [14]:
def top_k(canciones=(), artistas=(), tags=(), usuario=None, texto=None, k=10, pesos=None, explicar=True, mostrar=True):
    w = np.array([(pesos or W)[s] for s in SENALES])
    semillas = gr.resolver_preferencia(canciones, artistas, tags, usuario, texto)
    u = next((n for n in semillas if gr.tipo.get(n) == 'User'), None)
    cands = gr.candidatos(semillas, N_CANDIDATOS)
    F, extra = matriz_senales(semillas, cands, usuario=u, texto=texto)
    score = F @ w
    elegidos = top_indices(cands, score, k)
    puesto_grafo = {i: p for p, i in enumerate(np.argsort(-F[:, 0], kind='stable'), 1)}   # orden solo por G
    filas = []
    for p, i in enumerate(elegidos, 1):
        t = cands[i][0]
        filas.append({'canción': gr.nombre[t], 'artista': gr.nombre[gr.artista_de[t]], 'score': round(float(score[i]), 3),
                      **{s: round(float(F[i, j]), 2) for j, s in enumerate(SENALES)},
                      'estrellas': round(float(extra['estrellas (V)'][i]), 2), 'reseñas': int(extra['reseñas'][i]),
                      'Δ vs grafo': puesto_grafo.get(i, len(cands)) - p,
                      **({'por qué (grafo)': gr.explicar(semillas, t)} if explicar else {}), 'uri': t})
    df = pd.DataFrame(filas, index=pd.RangeIndex(1, len(filas) + 1, name='rank'))
    if mostrar:
        print('Preferencia:', '; '.join(gr.etiqueta(n) for n in list(semillas)[:5]), '…' if len(semillas) > 5 else '')
        print('Pesos:', {s: float(x) for s, x in zip(SENALES, w)}, '| perfil textual:',
              'sí' if perfil(semillas, texto=texto) is not None else 'no (A constante)')
    return df

### 5.1 Un usuario de CritiqueBrainz

In [15]:
usuarios = Counter(f['usuario'] for f in RESENAS.values())
usuario_demo = next(u for u, n in usuarios.most_common()
                    if 5 <= n <= 30 and sum(r in EMB for r in RESENAS_USUARIO[u]) >= 5)   # con reseñas escritas
display(top_k(usuario=usuario_demo).drop(columns='uri'))

Preferencia: usuario Jaime Gill 
Pesos: {'G': 0.2, 'V': 0.1, 'A': 0.7, 'E': 0.0} | perfil textual: sí


,canción,artista,score,G,V,A,E,estrellas,reseñas,Δ vs grafo,por qué (grafo)
rank,,,,,,,,,,,
1,Paper House,Associates,0.813,0.75,0.26,0.91,1.0,4.05,1,2,usuario Jaime Gill ─REVIEWED─ álbum «Singles» ─RELEASED_BY─ Associates ─SUNG_BY─ «Paper House» (Associates)
2,Lately,Massive Attack,0.789,0.43,0.60,0.92,0.0,4.28,3,128,usuario Jaime Gill ─REVIEWED─ álbum «Heligoland» ─RELEASED_BY─ Massive Attack ─SUNG_BY─ «Lately» (Massive ...
3,I Am Made Of Chalk,Crystal Castles,0.760,0.05,0.51,1.00,0.0,4.21,3,185,usuario Jaime Gill ─REVIEWED─ álbum «Heligoland» ─REVIEWED─ usuario egecelikci ─REVIEWED─ álbum «Crystal C...
4,Xxzxcuzx,Crystal Castles,0.760,0.05,0.51,1.00,0.0,4.21,3,185,usuario Jaime Gill ─REVIEWED─ álbum «Heligoland» ─REVIEWED─ usuario egecelikci ─REVIEWED─ álbum «Crystal C...
5,The Chaos,The Futureheads,0.714,0.60,0.12,0.83,1.0,3.96,1,40,usuario Jaime Gill ─REVIEWED─ álbum «This Is Not the World» ─RELEASED_BY─ The Futureheads ─SUNG_BY─ «The C...
6,This Is The Life,The Futureheads,0.714,0.60,0.12,0.83,1.0,3.96,1,40,usuario Jaime Gill ─REVIEWED─ álbum «This Is Not the World» ─RELEASED_BY─ The Futureheads ─SUNG_BY─ «This ...
7,The National Front Disco,Morrissey,0.712,0.51,0.04,0.87,0.0,3.90,1,90,usuario Jaime Gill ─REVIEWED─ álbum «Ringleader of the Tormentors» ─RELEASED_BY─ Morrissey ─SUNG_BY─ «The ...
8,From The Ritz To The Rubble,Arctic Monkeys,0.709,0.44,0.60,0.80,0.0,4.27,2,120,usuario Jaime Gill ─REVIEWED─ álbum «First Impressions of Earth» ─RELEASED_BY─ The Strokes ─SUNG_BY─ «Juic...
9,The First Of The Gang To Die,Morrissey,0.705,0.52,0.13,0.84,0.0,3.96,1,87,usuario Jaime Gill ─REVIEWED─ álbum «Ringleader of the Tormentors» ─RELEASED_BY─ Morrissey ─SUNG_BY─ «The ...


### 5.2 Una canción con reseñas (hay perfil textual)

In [16]:
salientes = Counter(f['s'] for f in gr.datos['SIMILAR_TO'])
cancion_demo = max((t for t in VEC if salientes[t]), key=lambda t: salientes[t])
tk = top_k(canciones=[next(i for i, u in gr.track_uri.items() if u == cancion_demo)], explicar=False)
display(tk.drop(columns='uri'))

Preferencia: «Everyone's At It» (Lily Allen) 
Pesos: {'G': 0.2, 'V': 0.1, 'A': 0.7, 'E': 0.0} | perfil textual: sí


,canción,artista,score,G,V,A,E,estrellas,reseñas,Δ vs grafo
rank,,,,,,,,,,
1,Back To The Start,Lily Allen,0.977,1.00,0.77,1.00,1.0,4.10,1,0
2,22,Lily Allen,0.885,0.54,0.77,1.00,1.0,4.10,1,10
3,From The Ritz To The Rubble,Arctic Monkeys,0.522,0.57,1.00,0.44,1.0,4.27,2,7
4,Paris Is Burning,Ladyhawke,0.516,0.64,0.88,0.43,1.0,4.19,1,0
5,4 In The Morning,Gwen Stefani,0.505,0.87,0.73,0.37,1.0,4.07,0,-3
6,Still Take You Home,Arctic Monkeys,0.476,0.34,1.00,0.44,1.0,4.27,2,37
7,Oh My,Ladyhawke,0.470,0.41,0.88,0.43,1.0,4.19,1,29
8,Foundations,Kate Nash,0.468,0.69,0.73,0.37,0.0,4.07,0,-5
9,I've Got A Secret,Kate Nash,0.460,0.31,0.65,0.48,0.0,4.01,1,45


### 5.3 Un artista, comparado con el ranking que usa solo el grafo

La comparación muestra el efecto de las preferencias. `V` baja las canciones cuyas reseñas están por debajo de la media
y sube las bien valoradas; `A` reordena según la afinidad con las reseñas del artista.

In [17]:
final = top_k(artistas=['Aerosmith'], explicar=False)
solo_grafo = top_k(artistas=['Aerosmith'], pesos={'G': 1.0, 'V': 0.0, 'A': 0.0, 'E': 0.0}, explicar=False, mostrar=False)
display(pd.concat({'ranking final': final[['canción', 'artista', 'estrellas', 'reseñas']],
                   'solo grafo (punto 3)': solo_grafo[['canción', 'artista', 'estrellas', 'reseñas']]}, axis=1))

Preferencia: Aerosmith 
Pesos: {'G': 0.2, 'V': 0.1, 'A': 0.7, 'E': 0.0} | perfil textual: no (A constante)


ranking final                                          solo grafo (punto 3)                                      
                  canción             artista estrellas reseñas                 canción             artista estrellas reseñas
rank                                                                                                                         
1                   Angel           Aerosmith      4.07       0           Sweet Emotion           Aerosmith      3.79       2
2           Milkcow Blues           Aerosmith      4.07       0         Round And Round           Aerosmith      3.72       1
3             Rock Steady         Bad Company      4.07       0             Rock Steady         Bad Company      4.07       0
4              Down On Me        Janis Joplin      4.07       0              Down On Me        Janis Joplin      4.07       0
5      Time Is On My Side  The Rolling Stones      4.07       0      Time Is On My Side  The Rolling Stones      4.07       0
6           Bitch Is Back           L.A. Guns      4.07       0           Bitch Is Back           L.A. Guns      4.07       0
7        Can't Get Enough         Bad Company      4.07       0        Can't Get Enough         Bad Company      4.07       0
8      Catcher In The Rye          Top Billin      4.06       1      Catcher In The Rye          Top Billin      4.06       1
9      If This Is The End     Jeff Scott Soto      4.07       0      If This Is The End     Jeff Scott Soto      4.07       0
10    The End Of The Line           Metallica      4.38       1  Mary Jane's Last Dance           Tom Petty      4.07       0

### 5.4 Texto libre (perfil textual = la consulta)

In [18]:
display(top_k(texto='rock de guitarras potente, con energía de concierto', explicar=False).drop(columns='uri'))

W1118 15:35:03.181000 28251 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Conexión perdida (ServiceUnavailable); reconectando 1/14 en 5s...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Preferencia: álbum «Members Only»; álbum «Up - Live From Chicago»; álbum «Culture Vultures»; álbum «B.R.M.C.»; álbum «Magic» …
Pesos: {'G': 0.2, 'V': 0.1, 'A': 0.7, 'E': 0.0} | perfil textual: sí


,canción,artista,score,G,V,A,E,estrellas,reseñas,Δ vs grafo
rank,,,,,,,,,,
1,Roofless,Avril,0.970,0.98,0.74,1.00,0.0,4.13,1,2
2,Room,Avril,0.970,0.98,0.74,1.00,0.0,4.13,1,2
3,Last To Die,Bruce Springsteen,0.873,0.86,0.72,0.90,0.0,4.12,1,5
4,Awake,Black Rebel Motorcycle Club,0.872,0.92,0.56,0.90,1.0,4.03,1,2
5,Northern Girl,Orson,0.836,0.97,0.49,0.85,0.0,3.99,1,0
6,Octagon,Pharaoh Overlord,0.835,1.00,0.67,0.81,0.0,4.09,1,-5
7,Even If It Breaks Your Heart,Will Hoge,0.787,0.98,0.73,0.74,1.0,4.12,1,-5
8,Drawing the Line (Album Version),Porcupine Tree,0.766,0.79,0.70,0.77,0.0,4.11,1,1
9,Great Expectations (Album Version),Porcupine Tree,0.766,0.79,0.70,0.77,0.0,4.11,1,1


## 6. Persistencia del Top-10 por usuario

El Top-10 final de cada usuario se guarda como `(:User)-[:RECOMMENDED {method: 'ranking_final', rank, score, ...}]->(:Track)`.
Las relaciones del punto 3 (`method: 'ppr+louvain'`, solo grafo) se conservan para poder compararlas.

In [19]:
t0 = time.time()
w = np.array([W[s] for s in SENALES])
filas = []
for u in sorted(usuarios):
    semillas = {u: 1.0}
    cands = gr.candidatos(semillas, N_CANDIDATOS)
    F, extra = matriz_senales(semillas, cands, usuario=u)
    score = F @ w
    for p, i in enumerate(top_indices(cands, score, 10), 1):
        filas.append({'u': u, 't': cands[i][0], 'rank': p, 'score': round(float(score[i]), 4),
                      'g': round(float(F[i, 0]), 3), 'v': round(float(extra['estrellas (V)'][i]), 3),
                      'a': round(float(F[i, 2]), 3), 'n': int(extra['reseñas'][i])})
print(f'{len(filas):,} recomendaciones para {len(usuarios)} usuarios en {time.time() - t0:.0f}s')

run_cypher("MATCH (:User)-[r:RECOMMENDED {method: 'ranking_final'}]->(:Track) DELETE r")
cargar_en_lotes('RECOMMENDED (ranking_final)', '''UNWIND $rows AS row
    MATCH (u:User {uri: row.u}) MATCH (t:Track {uri: row.t})
    CREATE (u)-[:RECOMMENDED {method: 'ranking_final', rank: row.rank, score: row.score,
                              graph_score: row.g, rating: row.v, text_affinity: row.a, n_reviews: row.n}]->(t)''', filas)

3,870 recomendaciones para 387 usuarios en 84s


✓ RECOMMENDED (ranking_final)                 3,870  (7.9s)


In [20]:
conteo = {r['m']: r['n'] for r in run_cypher('MATCH (:User)-[r:RECOMMENDED]->(:Track) RETURN r.method AS m, count(r) AS n')}
assert conteo.get('ranking_final') == len(filas), conteo
print('✓ RECOMMENDED por método:', conteo)
display(pd.DataFrame(run_cypher('''
    MATCH (u:User {uri: $u})-[r:RECOMMENDED {method: 'ranking_final'}]->(t:Track)-[:SUNG_BY]->(a:Artist)
    OPTIONAL MATCH (u)-[g:RECOMMENDED {method: 'ppr+louvain'}]->(t)
    RETURN r.rank AS rank, t.title AS cancion, a.name AS artista, r.score AS score, r.rating AS estrellas,
           r.n_reviews AS resenas, g.rank AS rank_solo_grafo
    ORDER BY rank''', u=usuario_demo)))

✓ RECOMMENDED por método: {'ranking_final': 3870, 'ppr+louvain': 3870}


,rank,cancion,artista,score,estrellas,resenas,rank_solo_grafo
0,1,Paper House,Associates,0.8133,4.053,1,3.0
1,2,Lately,Massive Attack,0.7894,4.276,3,NaN
2,3,I Am Made Of Chalk,Crystal Castles,0.7598,4.212,3,NaN
3,4,Xxzxcuzx,Crystal Castles,0.7598,4.212,3,NaN
4,5,The Chaos,The Futureheads,0.7142,3.959,1,NaN
5,6,This Is The Life,The Futureheads,0.7142,3.959,1,NaN
6,7,The National Front Disco,Morrissey,0.7120,3.902,1,NaN
7,8,From The Ritz To The Rubble,Arctic Monkeys,0.7087,4.272,2,NaN
8,9,The First Of The Gang To Die,Morrissey,0.7052,3.964,1,NaN
9,10,Girls Who Play Guitars,Maximo Park,0.6982,4.120,1,NaN


## 7. Resumen

| Componente | Decisión | Justificación |
|---|---|---|
| Candidatos | Top-200 de Personalized PageRank (punto 3), sin conocidas ni duplicados | El mejor generador en el punto 3 |
| `G` | `log(PPR)` escalado min-max | PPR tiene cola pesada: el logaritmo conserva el orden y reduce la dominancia del primer candidato |
| `V` | Promedio bayesiano (`K = 2`, μ global) de estrellas explícitas o estimadas desde el texto (ridge, confianza 0,5) | Multiplica por ~6 la cobertura de la opinión de usuarios. La estimación se valida con CV |
| `A` | Coseno entre el perfil de la preferencia y las reseñas del candidato | Validada como señal de relevancia (AUC en pares `SIMILAR_TO`) |
| `E` | Misma comunidad Louvain | Se evaluó: peso 0 en el ranking, pero se usa para explicar |
| Pesos | Rejilla (paso 0,1) con validación y prueba separadas; máximo `Q` con `R ≥ 1` | Incorporar preferencias sin perder relevancia; se reporta en prueba con ablación |
| Top-K | Orden por puntaje con máximo 2 canciones por artista | Diversidad, igual que en el punto 3 |

**Reproducibilidad.** Todas las semillas aleatorias están fijas: `SEMILLA = 42` para la división y las tareas, y la
semilla de Louvain del módulo. Todos los parámetros están definidos en la sección 0 y los pesos se recalculan con la
regla de la sección 4 en cada ejecución. El resultado queda en Neo4j como `RECOMMENDED {method: 'ranking_final'}`.

In [21]:
driver.close()